In [ ]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
data = pd.read_csv("synthetic_credit_card_fraud_dataset.csv")

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.shape

In [ ]:
data['ip_risk_score'].hist(bins=40, figsize=(8, 4))
plt.xlabel("IP Risk Score")
plt.ylabel("Frequency")
plt.title("Frequency of IP Risk Score")
plt.show()

In [ ]:
numeric_train = data.select_dtypes(include=[np.number])
corr_matrix = numeric_train.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(14, 14))
im = ax.imshow(corr_matrix, vmin=-1, vmax=1)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha="right")
ax.set_yticklabels(corr_matrix.columns)

for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        ax.text(
            j,
            i,
            f"{corr_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=9
        )

plt.title("Correlation Matrix with Numeric Values")
plt.tight_layout()
plt.show()

In [ ]:
def preprocess_pipeline_withip(data):
    X = data.drop(columns=['label','user_id', 'card_number', 'cvv','browser_Firefox', 'device_type_Desktop','time_of_day_Morning'])
    y = data["label"]

    
    dates = pd.to_datetime(data['expiry_date'], format='%m/%y')
    month = dates.dt.month
    year = dates.dt.year
    X["months_until_expiry"] = year + (month /12)
    X = X.drop(columns=['expiry_date'])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train[['ip_risk_score', 'months_until_expiry']]= scaler.fit_transform(X_train[['ip_risk_score', 'months_until_expiry']])
    X_test[['ip_risk_score', 'months_until_expiry']]= scaler.transform(X_test[['ip_risk_score', 'months_until_expiry']])

    return X_train, X_test, y_train, y_test
        

In [ ]:
def preprocess_pipeline_withoutip(data):
    X_noip = data.drop(columns=['label','user_id', 'card_number', 'cvv', 'ip_risk_score', 'browser_Firefox', 'device_type_Desktop','time_of_day_Morning'])
    y_noip = data["label"]

    
    dates = pd.to_datetime(data['expiry_date'], format='%m/%y')
    month = dates.dt.month
    year = dates.dt.year
    X_noip["months_until_expiry"] = year + (month /12)
    X_no_ip = X_noip.drop(columns=['expiry_date'])

    X_train_no_ip, X_test_no_ip, y_train_no_ip, y_test_no_ip = train_test_split(X_no_ip, y_noip, test_size=0.2, random_state=42, stratify=y_noip)

    scaler = StandardScaler()
    X_train_no_ip[['months_until_expiry']]= scaler.fit_transform(X_train_no_ip[['months_until_expiry']])
    X_test_no_ip[['months_until_expiry']]= scaler.transform(X_test_no_ip[['months_until_expiry']])

    return X_train_no_ip, X_test_no_ip, y_train_no_ip, y_test_no_ip

In [ ]:
preprocess_pipeline_withip(data)

In [ ]:
preprocess_pipeline_withoutip(data)

**Algorithm #1: Logistic Regression** 

In [ ]:
#With IP Risk Score
X_train, X_test, y_train, y_test = preprocess_pipeline_withip(data)

#Create Validation Set
X_train_lr, X_val_lr, y_train_lr, y_val_lr = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

#Add Bias Columns
X_train_bias = np.c_[np.ones((X_train_lr.shape[0], 1)), X_train_lr]
X_val_bias = np.c_[np.ones((X_val_lr.shape[0], 1)), X_val_lr]
X_test_bias = np.c_[np.ones((X_test.shape[0], 1)), X_test]

In [ ]:
# Without IP Risk Score
X_train_no_ip, X_test_no_ip, y_train_no_ip, y_test_no_ip = preprocess_pipeline_withoutip(data)

#Create Validation Set
X_train_no_ip_lr, X_val_no_ip_lr, y_train_no_ip_lr, y_val_no_ip_lr = train_test_split(
    X_train_no_ip,
    y_train_no_ip,
    test_size=0.25,
    random_state=42,
    stratify=y_train_no_ip
)

#Add Bias Columns
X_train_no_ip_bias = np.c_[np.ones((X_train_no_ip_lr.shape[0], 1)), X_train_no_ip_lr]
X_val_no_ip_bias = np.c_[np.ones((X_val_no_ip_lr.shape[0], 1)), X_val_no_ip_lr]
X_test_no_ip_bias = np.c_[np.ones((X_test_no_ip.shape[0], 1)), X_test_no_ip]

In [ ]:
#Logistic Function
def logistic_regression(t):
    return 1 / (1 + np.exp(-t))

In [ ]:
#Log Loss Cost Function
def log_loss(X, y, theta, epsilon=1e-7):
    p_hat = logistic_regression(X @ theta)
    return -np.mean(y * np.log(p_hat + epsilon) + (1 - y) * np.log(1 - p_hat + epsilon))

In [ ]:
#Gradients Function
def gradients(X, y, theta):
    m = len(y)
    p_hat = logistic_regression(X @ theta)
    return (1/m) * X.T @ (p_hat - y)

In [ ]:
#Batch Gradient Descent with IP Risk Score
eta = 0.1
n_epochs = 5000
record_every = 50
theta = np.random.default_rng(42).standard_normal(X_train_bias.shape[1])

epochs_recorded = []
train_losses = []
valid_losses = []

for epoch in range(n_epochs):
    grad = gradients(X_train_bias, y_train_lr, theta)
    theta = theta - eta * grad

    if epoch % record_every == 0:
        train_loss = log_loss(X_train_bias, y_train_lr, theta)
        val_loss = log_loss(X_val_bias, y_val_lr, theta)
        
        epochs_recorded.append(epoch)
        train_losses.append(train_loss)
        valid_losses.append(val_loss)

In [ ]:
#Batch Gradient Descent without IP Risk Score
eta = 0.1
n_epochs = 5000
record_every = 50

theta_no_ip = np.random.default_rng(42).standard_normal(X_train_no_ip_bias.shape[1])

epochs_recorded_no_ip = []
train_losses_no_ip = []
valid_losses_no_ip = []

for epoch in range(n_epochs):
    grad = gradients(X_train_no_ip_bias, y_train_no_ip_lr, theta_no_ip)
    theta_no_ip = theta_no_ip - eta * grad

    if epoch % record_every == 0:
        train_loss = log_loss(X_train_no_ip_bias, y_train_no_ip_lr, theta_no_ip)
        val_loss = log_loss(X_val_no_ip_bias, y_val_no_ip_lr, theta_no_ip)

        epochs_recorded_no_ip.append(epoch)
        train_losses_no_ip.append(train_loss)
        valid_losses_no_ip.append(val_loss)

In [ ]:
#Learning Curves with IP Risk Score
plt.figure(figsize=(8, 5))

plt.plot(epochs_recorded, train_losses, label="Training Loss")
plt.plot(epochs_recorded, valid_losses, label="Validation Loss")

plt.yscale("log")

plt.xlabel("Epoch")
plt.ylabel("Log-loss on Logarithmic Scale")
plt.title("Batch Gradient Descent: Training and Validation Loss with IP Risk Score")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
#Learning Curves without IP Risk Score
plt.figure(figsize=(8, 5))

plt.plot(epochs_recorded_no_ip, train_losses_no_ip, label="Training Loss")
plt.plot(epochs_recorded_no_ip, valid_losses_no_ip, label="Validation Loss")

plt.yscale("log")

plt.xlabel("Epoch")
plt.ylabel("Log-loss on Logarithmic Scale")
plt.title("Batch Gradient Descent: Training and Validation Loss without IP Risk Score")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
#Early Stopping Logic with IP Risk Score
best_loss = np.inf
best_theta = None
best_epoch = 0
patience = 100

theta = np.random.default_rng(42).standard_normal(X_train_bias.shape[1])

epochs_recorded = []
train_losses = []
valid_losses = []

epochs_without_improvement = 0

for epoch in range(n_epochs):
    grad = gradients(X_train_bias, y_train_lr, theta)
    theta = theta - eta * grad
    
    val_loss = log_loss(X_val_bias, y_val_lr, theta)
    
    if val_loss < best_loss:
        best_loss = val_loss
        best_theta = theta.copy()
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    
    if epoch % record_every == 0:
        epochs_recorded.append(epoch)
        train_losses.append(log_loss(X_train_bias, y_train_lr, theta))
        valid_losses.append(val_loss)
    
    if epochs_without_improvement >= patience:
        break

theta = best_theta.copy()

print("With IP Risk Score:")
print("Best Epoch:", best_epoch)
print("Best Loss:", best_loss)

In [ ]:
# Early Stopping Logic without IP Risk Score
best_loss_no_ip = np.inf
best_theta_no_ip = None
best_epoch_no_ip = 0
patience = 100

theta_no_ip = np.random.default_rng(42).standard_normal(X_train_no_ip_bias.shape[1])

epochs_recorded_no_ip = []
train_losses_no_ip = []
valid_losses_no_ip = []

epochs_without_improvement = 0

for epoch in range(n_epochs):
    grad = gradients(X_train_no_ip_bias, y_train_no_ip_lr, theta_no_ip)
    theta_no_ip = theta_no_ip - eta * grad
    
    val_loss = log_loss(X_val_no_ip_bias, y_val_no_ip_lr, theta_no_ip)
    
    if val_loss < best_loss_no_ip:
        best_loss_no_ip = val_loss
        best_theta_no_ip = theta_no_ip.copy()
        best_epoch_no_ip = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    
    if epoch % record_every == 0:
        epochs_recorded_no_ip.append(epoch)
        train_losses_no_ip.append(log_loss(X_train_no_ip_bias, y_train_no_ip_lr, theta_no_ip))
        valid_losses_no_ip.append(val_loss)
    
    if epochs_without_improvement >= patience:
        break

theta_no_ip = best_theta_no_ip.copy()

print("Without IP Risk Score:")
print("Best Epoch:", best_epoch_no_ip)
print("Best Loss:", best_loss_no_ip)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(epochs_recorded, train_losses, label="Training Loss")
plt.plot(epochs_recorded, valid_losses, label="Validation Loss")

plt.axvline(
    best_epoch,
    color="k",
    ls="--",
    lw=1,
    label=f"best epoch ({best_epoch})"
)

plt.yscale("log")

plt.xlabel("Epoch")
plt.ylabel("Log-loss on Logarithmic Scale")
plt.title("Batch Gradient Descent: Training and Validation Loss with IP Risk Score")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(epochs_recorded_no_ip, train_losses_no_ip, label="Training Loss")
plt.plot(epochs_recorded_no_ip, valid_losses_no_ip, label="Validation Loss")

plt.axvline(
    best_epoch_no_ip,
    color="k",
    ls="--",
    lw=1,
    label=f"best epoch ({best_epoch_no_ip})"
)

plt.yscale("log")

plt.xlabel("Epoch")
plt.ylabel("Log-loss on Logarithmic Scale")
plt.title("Batch Gradient Descent: Training and Validation Loss without IP Risk Score")

plt.legend()
plt.grid(True)

plt.show()

**Algorithm #2: Random Forest** 

In [ ]:
#Testing and training sets with and without IP risk score feature
X_train, X_test, y_train, y_test = preprocess_pipeline_withip(data)
X_train_no_ip, X_test_no_ip, y_train_no_ip, y_test_no_ip = preprocess_pipeline_withoutip(data)

In [ ]:
#Creating random forest model
from sklearn.ensemble import RandomForestClassifier
randomForest = RandomForestClassifier(random_state = 42)

In [ ]:
#Validation + hyperparameters: Stratified K-fold, Grid Search, and Randomized Search
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
#K-fold
skfold = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

skfold_scores = cross_val_score(
    randomForest,
    X_train,
    y_train,
    cv = skfold,
    scoring = "recall",
    n_jobs = -1
)
skfold_scores_no_ip = cross_val_score(
    randomForest,
    X_train_no_ip,
    y_train_no_ip,
    cv = skfold,
    scoring = "recall",
    n_jobs = -1
)
print("K-fold scores with IP:")
print(skfold_scores)
print("K-fold scores without IP:")
print(skfold_scores_no_ip)

In [ ]:
#K-fold visualization
plt.figure()
plt.plot(range(1, 6), skfold_scores, label = "With IP")
plt.plot(range(1, 6), skfold_scores_no_ip, label = "Without IP")
plt.grid(True)
plt.xlabel("Folds")
plt.ylabel("Recall score")
plt.title("Cross Validation Recall for K-fold")
plt.legend()
plt.show()

In [ ]:
#Grid search
parameters_gs = {
    "n_estimators" : [100, 200, 500],
    "max_depth": [None, 5, 10, 20],
    "max_features": ['sqrt', 'log2'],
    "min_samples_split": [2, 5, 10], #prevents overfitting
    "min_samples_leaf": [1, 2, 4] #prevents overfitting
}

gridSearch = GridSearchCV(
    randomForest,
    param_grid = parameters_gs,
    cv = skfold,
    scoring = "recall",
    n_jobs = -1
)
gridSearch_no_ip = GridSearchCV(
    randomForest,
    param_grid = parameters_gs,
    cv = skfold,
    scoring = "recall",
    n_jobs = -1
)
gridSearch.fit(X_train, y_train)
gridSearch_no_ip.fit(X_train_no_ip, y_train_no_ip)

In [ ]:
#Grid Search results:
print("-----Best parameters for grid search-----")
print("With IP:")
print(gridSearch.best_params_)
print("Without IP:")
print(gridSearch_no_ip.best_params_)
print("\n-----Best recall score-----")
print("With IP: ", gridSearch.best_score_)
print("Without IP: ", gridSearch_no_ip.best_score_)

In [ ]:
#Randomized Search
parameters_rs = {
    "n_estimators": range(100, 501, 50),
    "max_depth": range(5,20),
    "max_features": ['sqrt', 'log2'],
    "min_samples_split": range(2, 10), #prevents overfitting
    "min_samples_leaf": range(1, 10) #prevents overfitting
}

randomSearch = RandomizedSearchCV(
    randomForest,
    param_distributions = parameters_rs,
    n_iter = 50,
    cv = skfold,
    scoring = "recall",
    random_state = 42,
    n_jobs = -1
)
randomSearch_no_ip = RandomizedSearchCV(
    randomForest,
    param_distributions = parameters_rs,
    n_iter = 50,
    cv = skfold,
    scoring = "recall",
    random_state = 42,
    n_jobs = -1
)

randomSearch.fit(X_train, y_train)
randomSearch_no_ip.fit(X_train_no_ip, y_train_no_ip)

In [ ]:
#Randomized Search results
print("-----Best Random Parameters-----")
print("With IP:")
print(randomSearch.best_params_)
print("Without IP:")
print(randomSearch_no_ip.best_params_)

print("\n-----Best Random CV Score-----")
print("With IP:\t", randomSearch.best_score_)
print("Without IP:\t", randomSearch_no_ip.best_score_)

RSResults = pd.DataFrame(randomSearch.cv_results_)
topScores = RSResults.nlargest(10, "mean_test_score")
RSResults_no_ip = pd.DataFrame(randomSearch_no_ip.cv_results_)
topScores_no_ip = RSResults_no_ip.nlargest(10, "mean_test_score")

print("\n-----Top average scores-----")
print("With IP:")
print(topScores[["mean_test_score", "params"]])
print("\nWithout IP:")
print(topScores_no_ip[["mean_test_score", "params"]])

In [ ]:
#Testing both models
bestGrid = gridSearch.best_estimator_
bestRan = randomSearch.best_estimator_

bestGrid_no_ip = gridSearch_no_ip.best_estimator_
bestRan_no_ip = randomSearch_no_ip.best_estimator_

y_predict_grid = bestGrid.predict(X_test)
y_predict_ran = bestRan.predict(X_test)

y_predict_grid_no_ip = bestGrid_no_ip.predict(X_test_no_ip)
y_predict_ran_no_ip = bestRan_no_ip.predict(X_test_no_ip)

In [ ]:
#Grid search Recall scores and Confusion Matrices with IP risk feature
from sklearn.metrics import recall_score, confusion_matrix, ConfusionMatrixDisplay
print("Grid search results:")
print("Recall score:", recall_score(y_test, y_predict_grid))
print("Confusion matrix:")
ConfusionMatrixDisplay.from_estimator(bestGrid, X_test, y_test)

In [ ]:
#Random search Recall scores and Confusion Matrices with IP risk feature
print("Random search results:")
print("Recall score:", recall_score(y_test, y_predict_ran))
print("Confusion matrix:")
ConfusionMatrixDisplay.from_estimator(bestRan, X_test, y_test)

In [ ]:
#Grid search Recall scores and Confusion Matrices without IP risk feature
print("Grid search results:")
print("Recall score:", recall_score(y_test_no_ip, y_predict_grid_no_ip))
print("Confusion matrix:")
ConfusionMatrixDisplay.from_estimator(bestGrid_no_ip, X_test_no_ip, y_test_no_ip)

In [ ]:
#Random search Recall scores and Confusion Matrices without IP risk feature
print("Random search results:")
print("Recall score:", recall_score(y_test_no_ip, y_predict_ran_no_ip))
print("Confusion matrix:")
ConfusionMatrixDisplay.from_estimator(bestRan_no_ip, X_test_no_ip, y_test_no_ip)

In [ ]:
#Evaluate and Build Confusion Matrix

y_predict_lr = (logistic_regression(X_test_bias @ theta) >= 0.5).astype(int)
y_predict_lr_no_ip = (logistic_regression(X_test_no_ip_bias @ theta_no_ip) >= 0.5).astype(int)

In [ ]:
#Random logistic regression scores and Confusion Matrices with IP risk feature
print("Logistic Regression Results:")
print("Recall score:", recall_score(y_test, y_predict_lr))
print("Confusion matrix:")
ConfusionMatrixDisplay.from_predictions(y_test, y_predict_lr)

In [ ]:
#Random search Recall scores and Confusion Matrices without IP risk feature
print("Logistic regression results:")
print("Recall score:", recall_score(y_test_no_ip, y_predict_lr_no_ip))
print("Confusion matrix:")
ConfusionMatrixDisplay.from_predictions(y_test_no_ip, y_predict_lr_no_ip)

# AdaBoost Model

AdaBoost combines weak decision trees sequentially and gives more attention to samples that were misclassified by earlier trees.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


## Create the Base AdaBoost Model

A decision stump is a small decision tree with one level. `n_estimators` sets the number of stumps, and `learning_rate` controls how strongly each stump contributes to the final model.

In [ ]:
X_train, X_test, y_train, y_test = preprocess_pipeline_withip(data)

stump = DecisionTreeClassifier(max_depth=1, random_state=42)

ada_model = AdaBoostClassifier(
    estimator=stump,
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)


## Cross-Validation

Five-fold cross-validation evaluates recall because this project prioritizes minimizing missed fraudulent transactions.

In [ ]:
cv_scores = cross_val_score(
    ada_model,
    X_train,
    y_train,
    cv=5,
    scoring="recall"
)

print("Recall scores:", cv_scores)
print("Mean recall:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())


## Hyperparameter Tuning

Grid Search tests combinations of `n_estimators`, `learning_rate`, and the maximum depth of each decision-tree estimator to find the settings with the best cross-validation recall.

In [ ]:
param_grid = {
    "n_estimators": [25, 50, 100, 200],
    "learning_rate": [0.01, 0.1, 1.0],
    "estimator__max_depth": [1, 2, 3]
}

grid_search = GridSearchCV(
    estimator=ada_model,
    param_grid=param_grid,
    cv=5,
    scoring="recall",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation recall:", grid_search.best_score_)


## Final Test Evaluation

The selected model is evaluated once on the held-out test set.

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


## Confusion Matrix

False negatives represent fraudulent transactions missed by the model.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
display_labels = ["Legitimate", "Fraud"]

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)
disp.plot()
plt.show()

tn, fp, fn, tp = cm.ravel()
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)


## Without IP Risk Score

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


## Create the Base AdaBoost Model

In [ ]:
X_train, X_test, y_train, y_test = preprocess_pipeline_withoutip(data)

stump = DecisionTreeClassifier(max_depth=1, random_state=42)

ada_model = AdaBoostClassifier(
    estimator=stump,
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)


## Cross-Validation

In [ ]:
cv_scores = cross_val_score(
    ada_model,
    X_train,
    y_train,
    cv=5,
    scoring="recall"
)

print("Recall scores:", cv_scores)
print("Mean recall:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())


## Hyperparameter Tuning

In [ ]:
param_grid = {
    "n_estimators": [25, 50, 100, 200],
    "learning_rate": [0.01, 0.1, 1.0],
    "estimator__max_depth": [1, 2, 3]
}

grid_search = GridSearchCV(
    estimator=ada_model,
    param_grid=param_grid,
    cv=5,
    scoring="recall",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation recall:", grid_search.best_score_)


## Final Test Evaluation

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
display_labels = ["Legitimate", "Fraud"]

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)
disp.plot()
plt.show()

tn, fp, fn, tp = cm.ravel()
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)
